# Three qubits: Toffoli gates and teleportation

**Professor Nano Labs · Quantum computing**

**Try it live in your browser:** [Three qubits lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/three-qubits-teleportation/)
**Watch first:** [Two and three qubits quantum gates](https://www.youtube.com/watch?v=iMjpZwISIlA)
**Watch later:** [Single qubit and its logic gates](https://www.youtube.com/watch?v=rD_fH7O-D5Y) · [Two qubit state: Separable vs Entangled vs Bell states](https://www.youtube.com/watch?v=4BRQsrh4oGw) · [Quantum entanglement; exposing its key features](https://www.youtube.com/watch?v=9vDyS2_Hufk)

Three qubits have eight basis states $|abc\rangle$, written with qubit 1 first. A state is a list of eight amplitudes, and a gate is an $8\times8$ unitary matrix built from Kronecker products. The two sections match the lab's two tabs: gates as truth tables, then teleportation step by step. It needs only NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
I2 = np.eye(2)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]])
Z = np.diag([1, -1]).astype(complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
P0, P1 = np.diag([1, 0]).astype(complex), np.diag([0, 1]).astype(complex)

def kron3(a, b, c):
    return np.kron(np.kron(a, b), c)

def on(U, q):
    # a one-qubit gate U on qubit q (1, 2 or 3)
    ops = [I2, I2, I2]; ops[q - 1] = U
    return kron3(*ops)

def controlled(U, controls, target):
    # U on the target when every control qubit is 1
    proj = [I2, I2, I2]
    for c in controls: proj[c - 1] = P1
    hit = proj.copy(); hit[target - 1] = U - I2
    return np.eye(8) + kron3(*hit)

def cnot(c, t): return controlled(X, [c], t)
def toffoli(t): return controlled(X, [q for q in (1, 2, 3) if q != t], t)
def fredkin(c):
    a, b = [q for q in (1, 2, 3) if q != c]
    # a controlled swap is three CNOTs, the middle one controlled by c as well
    return cnot(b, a) @ controlled(X, [c, a], b) @ cnot(b, a)
CCZ = np.diag([1, 1, 1, 1, 1, 1, 1, -1]).astype(complex)

def basis(bits):
    s = np.zeros(8, complex); s[int(bits, 2)] = 1; return s

def show(state, label='state'):
    terms = [f'{c.real:+.3f}{c.imag:+.3f}i |{i:03b}>' if abs(c.imag) > 1e-9 else f'{c.real:+.3f} |{i:03b}>' for i, c in enumerate(state) if abs(c) > 1e-9]
    print(f'{label}: ' + '  '.join(terms))

## 1 · Gates as truth tables

CNOT, Toffoli and Fredkin only reorder the eight rows: each is a permutation matrix. Print where each row goes, then check that every gate undoes itself.

In [ ]:
def table(U, name):
    print(name)
    for i in range(8):
        j = int(np.argmax(np.abs(U[:, i])))
        print(f'  |{i:03b}> -> |{j:03b}>' + ('   (changed)' if j != i else ''))

table(toffoli(3), 'Toffoli, target 3')
table(fredkin(1), 'Fredkin, control 1')
gates = {'X1': on(X, 1), 'H2': on(H, 2), 'Z3': on(Z, 3), 'CNOT 1->3': cnot(1, 3), 'Toffoli -> 3': toffoli(3),
         'Fredkin 1': fredkin(1), 'CCZ': CCZ}
for name, U in gates.items():
    assert np.allclose(U.conj().T @ U, np.eye(8)) and np.allclose(U @ U, np.eye(8))
print('every gate is unitary and undoes itself')

# the half adder: Toffoli onto 3, then CNOT 1->2, on |a b 0>
for a in (0, 1):
    for b in (0, 1):
        s = cnot(1, 2) @ toffoli(3) @ basis(f'{a}{b}0')
        out = f'{int(np.argmax(np.abs(s))):03b}'
        print(f'{a} + {b}: output |{out}>  sum {out[1]}, carry {out[2]}')

In [ ]:
# every input at once: H on 1 and 2, then Toffoli
s = toffoli(3) @ on(H, 2) @ on(H, 1) @ basis('000')
show(s, 'H, H, Toffoli')

def bloch(state, q):
    # the Bloch vector of qubit q on its own, from the reduced density matrix
    rho = np.outer(state, state.conj()).reshape(2, 2, 2, 2, 2, 2)
    spec = {1: 'abcdbc->ad', 2: 'abcaec->be', 3: 'abcabf->cf'}[q]
    r = np.einsum(spec, rho)
    return np.real([np.trace(r @ X), np.trace(r @ Y), np.trace(r @ Z)])

print('arrow lengths:', [round(float(np.linalg.norm(bloch(s, q))), 3) for q in (1, 2, 3)])

# no cloning: CNOT copies 0 and 1, but not |+>
s = cnot(1, 2) @ on(H, 1) @ basis('000')
show(s, 'H, then CNOT 1->2')
two_copies = np.kron(np.kron(H @ [1, 0], H @ [1, 0]), [1, 0])
show(two_copies, 'two copies of |+> would be')

# phase kickback: qubit 3 in |->, Toffoli acts as CCZ on the rest
s = on(H, 3) @ toffoli(3) @ on(H, 3) @ on(H, 2) @ on(H, 1) @ basis('001')
show(s, 'phase kickback')

## 2 · Teleportation, step by step

Qubit 1 holds Alice's message $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$. H on qubit 2 and CNOT 2→3 make the pair $|\Phi^+\rangle=(|00\rangle+|11\rangle)/\sqrt2$, with qubit 3 going to Bob. Then CNOT 1→2 and H on qubit 1 give
$$\frac12\sum_{m_1,m_2}|m_1m_2\rangle\otimes X^{m_2}Z^{m_1}|\psi\rangle .$$
Alice measures qubits 1 and 2, sends the two bits $m_1m_2$, and Bob applies $X^{m_2}$ and then $Z^{m_1}$. To include a noisy pair, use density matrices: the pair becomes $v|\Phi^+\rangle\langle\Phi^+|+(1-v)I/4$.

In [ ]:
def message(theta, phi):
    return np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])

def reduced_bloch(rho, q):
    r = rho.reshape(2, 2, 2, 2, 2, 2)
    spec = {1: 'abcdbc->ad', 2: 'abcaec->be', 3: 'abcabf->cf'}[q]
    red = np.einsum(spec, r)
    return np.real([np.trace(red @ X), np.trace(red @ Y), np.trace(red @ Z)])

def teleport(psi, v=1.0, m=(0, 0), verbose=False):
    phi = np.array([1, 0, 0, 1]) / np.sqrt(2)
    pair = v * np.outer(phi, phi) + (1 - v) * np.eye(4) / 4
    rho = np.kron(np.outer(psi, psi.conj()), pair)                  # after step 2
    for U in (cnot(1, 2), on(H, 1)):                                # steps 3 and 4
        rho = U @ rho @ U.conj().T
    P = kron3([P0, P1][m[0]], [P0, P1][m[1]], I2)                   # step 5: Alice gets m
    rho = P @ rho @ P; p = np.real(np.trace(rho)); rho /= p
    fix = on(np.linalg.matrix_power(Z, m[0]), 3) @ on(np.linalg.matrix_power(X, m[1]), 3)
    rho = fix @ rho @ fix.conj().T                                  # step 6: Bob corrects
    if verbose:
        print(f'  m = {m}: probability {p:.3f}, Bob\'s arrow {reduced_bloch(rho, 3).round(3)}')
    bob = np.einsum('abcabf->cf', rho.reshape(2, 2, 2, 2, 2, 2))
    return p, np.real(psi.conj() @ bob @ psi)                      # outcome probability, fidelity

psi = message(np.radians(60), np.radians(45))
print('message arrow:', reduced_bloch(np.kron(np.outer(psi, psi.conj()), np.eye(4) / 4), 1).round(3))
for m in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    teleport(psi, 1.0, m, verbose=True)

vs = np.linspace(0, 1, 21)
F = [np.mean([teleport(psi, v, m)[1] for m in [(0, 0), (0, 1), (1, 0), (1, 1)]]) for v in vs]
plt.figure(figsize=(6, 3.4))
plt.plot(vs, F, 'o-', label='teleportation, computed')
plt.plot(vs, (1 + vs) / 2, 'k:', label='(1 + v)/2')
plt.axhline(2 / 3, color='#c47a12', ls='--', label='best without entanglement, 2/3')
plt.axvline(1 / 3, color='gray', lw=0.8)
plt.xlabel('pair quality v'); plt.ylabel('fidelity'); plt.legend(fontsize=8); plt.show()

## Check the lab's answers

These asserts confirm the statements in the lab's "Predict first" answers.

In [ ]:
rows = lambda s: [f'{i:03b}' for i in range(8) if abs(s[i]) > 1e-9]
# Q1: Toffoli on |110> gives |111>, and changes only those two rows
assert rows(toffoli(3) @ basis('110')) == ['111']
assert [i for i in range(8) if abs(toffoli(3)[i, i]) < 1e-9] == [6, 7]
# Q2: the half adder gives |a, a xor b, ab>; 1 + 1 -> |101>
for a in (0, 1):
    for b in (0, 1):
        assert rows(cnot(1, 2) @ toffoli(3) @ basis(f'{a}{b}0')) == [f'{a}{a ^ b}{a & b}']
# Q3: two Toffolis do nothing
assert np.allclose(toffoli(3) @ toffoli(3), np.eye(8))
# Q4: H, H, Toffoli: four rows of amplitude 1/2, qubit 3's arrow has length 1/2
s = toffoli(3) @ on(H, 2) @ on(H, 1) @ basis('000')
assert rows(s) == ['000', '010', '100', '111'] and np.allclose(np.abs(s[[0, 2, 4, 7]]), 0.5)
assert np.isclose(np.linalg.norm(bloch(s, 3)), 0.5)
# Q5: H then CNOT gives a Bell pair, not two copies of |+>
s = cnot(1, 2) @ on(H, 1) @ basis('000')
assert rows(s) == ['000', '110'] and np.isclose(np.linalg.norm(bloch(s, 1)), 0) and np.isclose(np.linalg.norm(bloch(s, 2)), 0)
# Q6: Alice's four results each have probability 1/4, for any message
for th, ph in [(0.3, 2.0), (2.5, -1.0), (np.pi / 2, 0)]:
    for m in [(0, 0), (0, 1), (1, 0), (1, 1)]:
        assert np.isclose(teleport(message(th, ph), 1.0, m)[0], 0.25)
# Q7: without the bits, Bob's qubit averaged over Alice's results is I/2: it passes a check half the time
psi = message(1.1, 0.7)
parts = [np.linalg.matrix_power(X, m2) @ np.linalg.matrix_power(Z, m1) @ psi for m1 in (0, 1) for m2 in (0, 1)]
avg = sum(np.outer(p, p.conj()) for p in parts) / 4
assert np.allclose(avg, np.eye(2) / 2) and np.isclose(np.real(psi.conj() @ avg @ psi), 0.5)
# Q8: fidelity (1 + v)/2 for any message; above 2/3 only when v > 1/3
for v in (0, 0.25, 0.5, 1):
    assert all(np.isclose(teleport(message(0.9, 2.2), v, m)[1], (1 + v) / 2) for m in [(0, 0), (1, 1)])
assert np.isclose((1 + 1 / 3) / 2, 2 / 3)
print('All checks passed.')

## Things to try

1. **Build the full adder.** Add a fourth line on paper: which Toffolis and CNOTs turn $|a,b,c_{\rm in},0\rangle$ into the sum and carry? Can you do it with three qubits if you allow garbage outputs?
2. **Teleport through a different Bell pair.** Start the pair as $|\Psi^-\rangle=(|01\rangle-|10\rangle)/\sqrt2$. Which correction must Bob apply for each of Alice's results now?
3. **Entanglement swapping.** Use four qubits: teleport qubit 2 of one Bell pair onto qubit 4. Qubits 1 and 4 end up entangled though they never met.
4. **Noise in the gates.** Replace Alice's CNOT with a slightly wrong one, $\mathrm{CNOT}^{0.95}$. How does the fidelity depend on the message?